In [1]:
import io 
import numpy as np 
import pandas as pd 

In [3]:
df = pd.read_csv('course_lead_scoring_2026.csv')

# Data preparation
cat_cols = ['lead_source', 'industry', 'employment_status', 'location']
num_cols = [
    'annual_income',
    'number_of_courses_viewed',
    'interaction_count',
    'lead_score',
]

for c in cat_cols:
  df[c] = df[c].fillna('NA')
for c in num_cols:
  df[c] = df[c].fillna(0.0)

print('Dataset shape:', df.shape)
df.head()

Dataset shape: (5000, 9)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Question 1: What is the most frequent observation (mode) for the column industry?

In [4]:
industry_mode = df['industry'].mode()[0]
print(f'Most frequent industry (mode): {industry_mode}')

Most frequent industry (mode): technology


# Question 2: Correlation matrix for numerical features

In [5]:
num_all = num_cols + ['converted']
corr_matrix = df[num_all].corr()
print('Correlation Matrix:')
display(corr_matrix)

Correlation Matrix:


,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [6]:
from sklearn.metrics import mutual_info_score
from sklearn.model_selection import train_test_split

In [8]:
from sklearn.metrics import mutual_info_score
from sklearn.model_selection import train_test_split

# Split data according to instructions
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train['converted'].values
y_val = df_val['converted'].values

# Remove target from dataframes
df_train = df_train.drop(columns=['converted'])
df_val = df_val.drop(columns=['converted'])
df_test = df_test.drop(columns=['converted'])

## Question 3: Mutual information scores between converted and categorical variables

In [9]:
print('Mutual Information Scores:')
for c in cat_cols:
  score = mutual_info_score(df_train[c], y_train)
  print(f'{c}: {round(score, 2)}')

Mutual Information Scores:
lead_source: 0.03
industry: 0.0
employment_status: 0.02
location: 0.0


## Question 4 (Logistic Regression & Validation Accuracy)

In [10]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# One-hot encoding using DictVectorizer
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

# Train model with specified parameters
model = LogisticRegression(
    solver='liblinear', C=1.0, max_iter=1000, random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
acc = accuracy_score(y_val, y_pred)
print(f'Validation Accuracy (C=1.0): {round(acc, 2)} (exact: {acc})')

Validation Accuracy (C=1.0): 0.65 (exact: 0.645)


## Question 5: Feature elimination technique

In [11]:
orig_acc = acc
orig_cols = cat_cols + num_cols

print('Accuracy difference when dropping each feature:')
for col_to_drop in orig_cols:
  sub_train = df_train.drop(columns=[col_to_drop])
  sub_val = df_val.drop(columns=[col_to_drop])

  dv_sub = DictVectorizer(sparse=False)
  Xt = dv_sub.fit_transform(sub_train.to_dict(orient='records'))
  Xv = dv_sub.transform(sub_val.to_dict(orient='records'))

  m_sub = LogisticRegression(
      solver='liblinear', C=1.0, max_iter=1000, random_state=42
  )
  m_sub.fit(Xt, y_train)

  acc_sub = accuracy_score(y_val, m_sub.predict(Xv))
  diff = orig_acc - acc_sub
  print(f'Drop {col_to_drop:25}: accuracy = {acc_sub:.4f}, diff = {diff:.4f}')

Accuracy difference when dropping each feature:
Drop lead_source              : accuracy = 0.6420, diff = 0.0030
Drop industry                 : accuracy = 0.6450, diff = 0.0000
Drop employment_status        : accuracy = 0.6450, diff = 0.0000
Drop location                 : accuracy = 0.6450, diff = 0.0000
Drop annual_income            : accuracy = 0.7240, diff = -0.0790
Drop number_of_courses_viewed : accuracy = 0.6430, diff = 0.0020
Drop interaction_count        : accuracy = 0.6010, diff = 0.0440
Drop lead_score               : accuracy = 0.6440, diff = 0.0010


## Question 6: Regularized logistic regression with different C values

In [12]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]

print('Validation accuracy for different C values:')
for c_val in C_values:
  m_reg = LogisticRegression(
      solver='liblinear', C=c_val, max_iter=1000, random_state=42
  )
  m_reg.fit(X_train, y_train)
  acc_reg = accuracy_score(y_val, m_reg.predict(X_val))
  print(f'C = {c_val:<10}: accuracy = {round(acc_reg, 3)} (exact: {acc_reg})')

Validation accuracy for different C values:
C = 1e-06     : accuracy = 0.598 (exact: 0.598)
C = 1e-05     : accuracy = 0.598 (exact: 0.598)
C = 0.0001    : accuracy = 0.613 (exact: 0.613)
C = 0.001     : accuracy = 0.645 (exact: 0.645)
